### Packages for the Data Generation and Mopdelling of PD (Probability of Default), LGD (Loss Given Default) and EAD (Exposure at Default)

In [1]:
# Data Management and Processing
import pandas as pd
import numpy as np
import scipy
import random

In [2]:
# Machine Learning and Statistics
import sklearn
import statsmodels.api as sm
import tensorflow as tf

2025-03-21 13:17:25.279186: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Data generator

##### Support functions

In [3]:
############# Function to create a profession based on the educational level #########################
def generate_profession(education):
    if education == "high school or lower":
        return random.choices(["LowSkilled", "Unemployed_LowSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education == "ausbildung":
        return random.choices(["MediumSkilled", "Unemployed_MediumSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education in ["bachelor degree", "post graduate degree"]:
        return random.choices(["HighSkilled", "Unemployed_HighSkilled"], weights=[0.9, 0.1], k=1)[0]

In [15]:
############################### Function to generate monthly income and expenditure ##################################                

# Define income parameters for different profession levels and age ranges
income_parameters = {
    ("LowSkilled", "Unemployed_LowSkilled"): {
        (30, 35): {"mean": 1000, "std_dev": 200, "max_income": 2000},
        (36, 40): {"mean": 1200, "std_dev": 200, "max_income": 2400},
        (41, 45): {"mean": 1500, "std_dev": 250, "max_income": 3000},
        (46, 50): {"mean": 1800, "std_dev": 300, "max_income": 3600},
        (51, 55): {"mean": 2000, "std_dev": 350, "max_income": 4000},
        (56, 60): {"mean": 2200, "std_dev": 400, "max_income": 4300},
        (61, 65): {"mean": 2400, "std_dev": 600, "max_income": 4500},
    },
    ("MediumSkilled", "Unemployed_MediumSkilled"): {
        (30, 35): {"mean": 1800, "std_dev": 300, "max_income": 4000},
        (36, 40): {"mean": 2300, "std_dev": 400, "max_income": 5000},
        (41, 45): {"mean": 2600, "std_dev": 500, "max_income": 6000},
        (46, 50): {"mean": 3000, "std_dev": 600, "max_income": 7000},
        (51, 55): {"mean": 3500, "std_dev": 800, "max_income": 8000},
        (56, 60): {"mean": 4000, "std_dev": 800, "max_income": 9000},
        (61, 65): {"mean": 5000, "std_dev": 1000, "max_income": 10000},
    },
    ("HighSkilled", "Unemployed_HighSkilled"): {
        (30, 35): {"mean": 3000, "std_dev": 500, "max_income": 10000},
        (36, 40): {"mean": 4500, "std_dev": 700, "max_income": 15000},
        (41, 45): {"mean": 6000, "std_dev": 1000, "max_income": 20000},
        (46, 50): {"mean": 7000, "std_dev": 1500, "max_income": 25000},
        (51, 55): {"mean": 8000, "std_dev": 2000, "max_income": 30000},
        (56, 60): {"mean": 9000, "std_dev": 3000, "max_income": 40000},
        (61, 65): {"mean": 10000, "std_dev": 4000, "max_income": 50000},
    }
}

# To calculate the monthly income and expenditure
def generate_income_expense(profession_undertake, current_age, num_dependents):
    # Iterate over income parameters for each profession group
    for profession_group, age_ranges in income_parameters.items():
        # Check if profession_undertake is one of the professions in the profession_group tuple
        if profession_undertake in profession_group:
            # Iterate over the age ranges and income parameters
            for age_range, params in age_ranges.items():
                if age_range[0] <= current_age <= age_range[1]:
                    mean_income = params["mean"]
                    std_dev = params["std_dev"]
                    max_income = params["max_income"]
                    unemployement_money = mean_income * 0.5  # 50% of mean income for unemployment

                    # If profession_undertake contains the word "Unemployed" before "_", return the unemployment money
                    if profession_undertake.split("_")[0] == "Unemployed":
                        income = unemployement_money
                        expenditure = generate_expenditure(income, mean_income, num_dependents) # The belong to the population under mean_income
                        return income, expenditure
                    
                    # If profession_undertake does not contain the word "Unemployed", generate income with an specific rule
                    else:
                        calc_income = int(np.random.normal(mean_income, std_dev))
                        income = max(unemployement_money, min(calc_income, max_income))  
                        expenditure = generate_expenditure(income, mean_income, num_dependents)
                        return income, expenditure

# Support function to calculate the expenditure
def generate_expenditure(income, mean_income, num_dependents):
    # Values for low and for high income people (lower possible value, mode, higher possible value) depending on the number of dependents
    low_income_params = [(0.5, 0.7, 1.5), (0.7, 0.8, 1.5), (0.8, 0.9, 1.5), (0.9, 0.9, 1.5), (0.9, 1.0, 1.5)]
    high_income_params = [(0.5, 0.7, 1.5), (0.6, 0.7, 1.5), (0.7, 0.75, 1.5), (0.7, 0.75, 1.5), (0.8, 0.85, 1.5)]
    # The parameters that should be taken depend on whether the income is below or above the mean income
    params = low_income_params if income < mean_income else high_income_params
    # Here we recover the parameters
    left, mode, right = params[min(num_dependents, 4)]  # Ensure index stays within range
    # The expenditure is calculated given a rule of min, mode, max
    expenditure = income * np.random.triangular(left=left, mode=mode, right=right)
    return expenditure



In [16]:
############################### Function to generate the credit to be requested ##################################

In [17]:
############################### Function to generate the Y-variable (default 0, 1) ##################################     
def default_y_calculation(profession, debt_income_ratio):
    # past_credits, dependents, professon, debt_income_ratio
    if profession == ("Unemployed_LowSkilled" or "Unemployed_MediumSkilled") and debt_income_ratio > 0.1:
        return 1
    if profession == ("Unemployed_HighSkilled") and debt_income_ratio > 0.2:
        return 1     


##### Data Generator for the original state of individuals

In [37]:
# Function to generate data accordingly to some requirements
def data_generator(number_of_customers):
    data = []
    for i in range(number_of_customers):
        
        # ---------------- X-Variables -------------------------------#
        ##### Variables not directly dependent on other variables #####
        name = f"name{i}" # names are created according to the index "i"
        age = random.randint(30, 60) # As the maximum attainable age that we want in the game is 65
        education_level = random.choices(["high school or lower", "ausbildung", "bachelor degree", "post graduate degree"],  weights=[0.3, 0.3, 0.3, 0.1], k=1)[0]
        # Number of unpaid past credits
        past_credits = random.choices([0, 1, 2, 3], weights=[0.6, 0.3, 0.08, 0.02], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        # Number of dependents
        dependents = random.choices([0, 1, 2, 3, 4], weights=[0.6, 0.3, 0.06, 0.03, 0.01], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        
        ##### Variables directly dependent on other variables #####
        # Generate profession based on education level
        profession = generate_profession(education_level)
        # Generate monthly income based on profession, age and number of dependents
        monthly_income = generate_income_expense(profession, age, dependents)[0]
        # Generate monthly expenditure dependening on the income, mean income and number of dependents
        monthly_expenditure = generate_income_expense(profession, age, dependents)[1]
        
        ##### Other variables generated from the variables above #####
        savings_debt = monthly_income - monthly_expenditure
        # Debt to income ratio
        if savings_debt < 0:
            #debt_to_income_ratio = f"{abs(savings_debt/monthly_income):.2%}"
            debt_to_income_ratio = abs(savings_debt/monthly_income)
        else:
            #debt_to_income_ratio = f"{0:.2%}"
            debt_to_income_ratio = 0
        
        # ---------------- Credit amount requested and time of the request--------------------------------#
        
        
        # ---------------- Y-Variable --------------------------------#
        
        data.append({
            'name': name,
            'age': age,
            'educational level': education_level,
            'number of not paid past credits': past_credits,
            'dependents': dependents,
            'profession': profession,
            'monthly income': monthly_income,
            'monthly expenditure': monthly_expenditure,
            'savings (debt)': savings_debt,
            'debt-to-income ratio': debt_to_income_ratio
        })

    # Create a pandas DataFrame
    df = pd.DataFrame(data)
    return df

##### Generating one data frame

In [38]:
number_of_customers_1 = 1000
df_1 = data_generator(number_of_customers_1)
df_1

,name,age,educational level,number of not paid past credits,dependents,profession,monthly income,monthly expenditure,savings (debt),debt-to-income ratio
0,name0,35,bachelor degree,0,1,HighSkilled,3179.0,3739.484765,-560.484765,0.176309
1,name1,60,ausbildung,0,1,MediumSkilled,4283.0,2703.628404,1579.371596,0.000000
2,name2,47,bachelor degree,1,2,HighSkilled,7629.0,5223.686349,2405.313651,0.000000
3,name3,54,ausbildung,1,0,MediumSkilled,1750.0,2053.144859,-303.144859,0.173226
4,name4,33,post graduate degree,0,3,HighSkilled,3568.0,1871.672322,1696.327678,0.000000
...,...,...,...,...,...,...,...,...,...,...
995,name995,47,bachelor degree,1,0,HighSkilled,7062.0,5263.524466,1798.475534,0.000000
996,name996,55,bachelor degree,0,1,HighSkilled,8925.0,7832.404738,1092.595262,0.000000
997,name997,34,high school or lower,2,0,LowSkilled,885.0,1041.646042,-156.646042,0.177001
998,name998,50,high school or lower,0,1,Unemployed_LowSkilled,900.0,983.371828,-83.371828,0.092635


##### DF statistics

In [39]:
df_1.describe()

,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,44.729000,0.515000,0.575000,3535.920000,3381.028256,154.891744,0.128773
std,8.775338,0.725466,0.851528,2461.698064,2669.476219,1654.466393,0.264988
min,30.000000,0.000000,0.000000,500.000000,426.149080,-13268.141223,0.000000
25%,37.750000,0.000000,0.000000,1783.000000,1544.719415,-350.637373,0.000000
50%,44.000000,0.000000,0.000000,2668.000000,2470.867644,208.891412,0.000000
75%,53.000000,1.000000,1.000000,4657.250000,4343.047020,809.927801,0.155691
max,60.000000,3.000000,4.000000,13905.000000,18630.380448,9431.133687,2.832652


Monthly income by profession

In [40]:
df_1.groupby('profession')['monthly income'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,364.0,5923.008242,2443.365775,1522.0,3968.25,5623.0,7240.0,13905.0
LowSkilled,263.0,1624.688213,526.865557,500.0,1223.00,1567.0,1968.5,3162.0
MediumSkilled,273.0,2782.791209,931.330577,1115.0,2065.00,2532.0,3379.0,5555.0
Unemployed_HighSkilled,43.0,2930.232558,1006.416293,1500.0,2250.00,3000.0,4000.0,4500.0
Unemployed_LowSkilled,22.0,786.363636,218.317058,500.0,525.00,900.0,1000.0,1100.0
Unemployed_MediumSkilled,35.0,1418.571429,416.407202,900.0,900.00,1500.0,1750.0,2000.0


Debt-to-income ratio by profession

In [41]:
df_1.groupby('profession')['debt-to-income ratio'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,364.0,0.168707,0.341109,0.0,0.0,0.000000,0.207971,2.832652
LowSkilled,263.0,0.118704,0.238319,0.0,0.0,0.000000,0.173787,2.033971
MediumSkilled,273.0,0.108024,0.198978,0.0,0.0,0.000000,0.137005,0.946639
Unemployed_HighSkilled,43.0,0.048665,0.104764,0.0,0.0,0.000000,0.017139,0.415140
Unemployed_LowSkilled,22.0,0.101571,0.112407,0.0,0.0,0.068458,0.221907,0.297447
Unemployed_MediumSkilled,35.0,0.066467,0.106072,0.0,0.0,0.000000,0.087833,0.457267
